In [1]:
import pandas as pd
from uplot.tikz.pgfdata import to_pgfdata

csvkwargs = dict(sep=' ', float_format='%.6f', na_rep='nan')

In [2]:
stats, basedata, basevar, baseline = pd.read_pickle('res/stats.pickle')

In [3]:
dh = (basevar['mean'] - baseline['mean']).xs('h', level='param')
pd.concat({**basevar.xs('h', level='param')[['mean', 'std']], 'off': 100*dh, 'roff': dh/baseline.loc['h', 'std']}, axis='columns').xs(True, level='sm').sort_index()

mean       std       off      roff
calib_ver hf_ver    pc    late  highz                                        
B22-CCM89 B22-CCM89 pcorr False False  0.717273  0.015077  0.127029  0.086715
B22-F99   B22-F99   pcorr False False  0.713036  0.014576 -0.296623 -0.202484
T21-CCM89 T21-CCM89 pcorr False False  0.717030  0.015070  0.102758  0.070146
T21-F99   T21-F99   pcorr False False  0.713356  0.014566 -0.264657 -0.180663
nom       nom       pborg False False  0.710141  0.014747 -0.586146 -0.400123
                                True   0.706945  0.014863 -0.905752 -0.618297
                          True  False  0.721165  0.016783  0.516230  0.352396
                                True   0.724014  0.017400  0.801128  0.546877
                    pcorr False False  0.716003  0.014649  0.000000  0.000000
                                True   0.716717  0.015114  0.071478  0.048793
                          True  False  0.724202  0.016377  0.819910  0.559698
                                True   0.729609  0.017355  1.360643  0.928820
                    pcov  False False  0.711624  0.018240 -0.437832 -0.298879
                                True   0.706828  0.017381 -0.917441 -0.626276
                          True  False  0.719320  0.018962  0.331777  0.226482
                                True   0.716528  0.019052  0.052506  0.035842

In [4]:
off = stats['mean'] - baseline['mean']
open('paper/plotdata/res.tex', 'w').write(str(to_pgfdata((100 * stats.assign(
    idx=list(map('/'.join, zip(stats['dataname'], stats['variant']))),
    off=off, roff=off/(100*baseline['std'])
).xs('h', level='param').set_index('idx')[['mean', 'std', 'off', 'roff']]).to_dict(orient='index'), namespace='private')))

42093

In [5]:
dh = (basedata['mean'] - baseline['mean']).xs('h', level='param')
pd.concat({**basedata.xs('h', level='param')[['mean', 'std']], 'off': 100*dh, 'roff': dh/baseline.loc['h', 'std']}, axis='columns').sort_index()  #.xs('latz', level='z_mode').xs('AV', level='use_AV').sort_index()

mean       std       off      roff
R_mode  use_2x step                                         
distR   1x     False  0.716003  0.014649  0.000000  0.000000
               True   0.721603  0.014812  0.560069  0.382322
        2x     False  0.716949  0.014766  0.094664  0.064621
               True   0.721636  0.014569  0.563353  0.384564
globalR 1x     False  0.707778  0.014791 -0.822508 -0.561472
               True   0.714147  0.015164 -0.185525 -0.126646
        2x     False  0.711115  0.015012 -0.488812 -0.333679
               True   0.715948  0.014906 -0.005460 -0.003727

In [7]:
s = pd.concat({
    key: group[['mean', 'std']].assign(off=group['mean']-baseline.loc['h', 'mean'])
    for key, group in stats.loc['nom', 'nom', :, True].xs('h', level='param').xs('distR', level='R_mode').reset_index(level=['late', 'highz']).reorder_levels(['pc', 'step', 'use_2x']).groupby(['late', 'highz'])
}, axis='columns').sort_index(ascending=[True, True, True])

s = pd.concat({key: s.loc[key] for key in ('pcorr', 'pcov', 'pborg')}, axis='index', names=s.index.names)

s.index = pd.MultiIndex.from_tuples(((i[0], {False: 'nostep', True: 'step'}[i[1]], *i[2:]) for i in s.index), names=s.index.names)
s.columns = [{
    (False, False): 'full',
    (False, True): 'hz',
    (True, False): 'late',
    (True, True): 'latehz'
}[c[:2]]+'-'+c[2] for c in s.columns]
s.to_csv('paper/plotdata/stats.txt', **csvkwargs, index=True)
s

full-mean  full-std  full-off   hz-mean    hz-std  \
pc    step   use_2x                                                      
pcorr nostep 1x       0.716003  0.014649  0.000000  0.716717  0.015114   
             2x       0.716949  0.014766  0.000947  0.718005  0.015008   
      step   1x       0.721603  0.014812  0.005601  0.722317  0.015598   
             2x       0.721636  0.014569  0.005634  0.723710  0.015642   
pcov  nostep 1x       0.711624  0.018240 -0.004378  0.706828  0.017381   
             2x       0.713517  0.017700 -0.002486  0.710218  0.017377   
      step   1x       0.714730  0.017921 -0.001272  0.710199  0.017464   
             2x       0.715546  0.017695 -0.000456  0.710999  0.017044   
pborg nostep 1x       0.710141  0.014747 -0.005861  0.706945  0.014863   
             2x       0.711797  0.014855 -0.004206  0.708579  0.014260   
      step   1x       0.714536  0.015028 -0.001467  0.711124  0.015020   
             2x       0.715728  0.015002 -0.000275  0.712154  0.014933   

                       hz-off  late-mean  late-std  late-off  latehz-mean  \
pc    step   use_2x                                                         
pcorr nostep 1x      0.000715   0.724202  0.016377  0.008199     0.729609   
             2x      0.002003   0.720283  0.015754  0.004280     0.726544   
      step   1x      0.006314   0.728830  0.016976  0.012827     0.734282   
             2x      0.007707   0.724023  0.016250  0.008020     0.729558   
pcov  nostep 1x     -0.009174   0.719320  0.018962  0.003318     0.716528   
             2x     -0.005785   0.718950  0.019392  0.002948     0.715130   
      step   1x     -0.005804   0.721594  0.019153  0.005592     0.718491   
             2x     -0.005004   0.718043  0.019476  0.002040     0.716348   
pborg nostep 1x     -0.009058   0.721165  0.016783  0.005162     0.724014   
             2x     -0.007423   0.716342  0.015740  0.000339     0.720881   
      step   1x     -0.004879   0.725164  0.017143  0.009161     0.726986   
             2x     -0.003849   0.719672  0.016773  0.003669     0.723217   

                     latehz-std  latehz-off  
pc    step   use_2x                          
pcorr nostep 1x        0.017355    0.013606  
             2x        0.016950    0.010542  
      step   1x        0.016820    0.018279  
             2x        0.016260    0.013555  
pcov  nostep 1x        0.019052    0.000525  
             2x        0.018621   -0.000873  
      step   1x        0.019229    0.002488  
             2x        0.018800    0.000346  
pborg nostep 1x        0.017400    0.008011  
             2x        0.016831    0.004879  
      step   1x        0.017277    0.010983  
             2x        0.016836    0.007214